In [ ]:
"""Follows similar analysis structure as FlowCurves 1_0 and 1_1"""

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import re

In [ ]:
base_dir = Path(r"C:\Users\myles\OneDrive\Desktop\adhesive_1_1_data_long")  # contains data_1_1_phi*
folders = sorted(base_dir.glob("data_1_1_phi*"))


In [ ]:
def find_table_start(path):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Time"):
                return i
    raise ValueError(f"No table header found in {path}")

def find_table_end(path):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Loop"):
                return i
    return None

In [ ]:
columns = []
column_names = []

for folder in folders:
    # phi from folder name: data_0_0_phi0.55
    mphi = re.search(r"phi([0-9.]+)", folder.name)
    if not mphi:
        continue
    phi_str = mphi.group(1)

    run_files = sorted(folder.glob("run_*"))  # e.g. run_0.01.txt etc.
    print(folder.name, "->", len(run_files), "run files")

    for file in run_files:
        start = find_table_start(file)
        end = find_table_end(file)

        df = pd.read_csv(
            file,
            delim_whitespace=True,
            skiprows=start,
            nrows=(end - start) if end is not None else None,
            on_bad_lines="skip",
            engine="python"
        )

        col = df.iloc[:, 4]  # 5th column (stress_xy)
        
    

        # change filename: run_0.006.txt -> 0.006
        a_str = file.stem.replace("run_", "")



        new_name = f"phi{phi_str}_a{a_str}_stress_xy"
        col = col.rename(new_name)

        columns.append(col)
        column_names.append(new_name)

new_df = pd.concat(columns, axis=1)
new_df.columns = column_names
new_df.insert(0, "Time", range(len(new_df)))

print("Final dataframe shape:", new_df.shape)


new_df.head()

In [ ]:

plt.figure(figsize=(12, 8))
for i, col in enumerate(new_df.columns[1:]):  # skip Time column
    if i % 14 == 0 and i > 0:
        plt.xlabel("Time")
        plt.ylabel("Reduced Viscosity")
        plt.legend()
        plt.yscale("log")
        plt.show()
        plt.figure(figsize=(12, 8))
    plt.plot(new_df["Time"], np.abs(new_df[col]/ (0.01*0.1)), label=col)
plt.xlabel("Time")
plt.ylabel("Reduced Viscosity")
plt.title("Reduced Viscosity vs Time for Different phi and a")
plt.legend()
plt.yscale("log")
plt.show()

In [ ]:
eta_f = 0.1      # fluid viscosity
gamma_dot = 0.01 # shear rate

# steady state stress is between time 100 and the end
steady_state_stresses = []
for col in new_df.columns[1:]:  # skip Time column
    steady_state_stress = np.abs(new_df[col][100:].mean())  # average stress from time 100 to end
    steady_state_stresses.append(steady_state_stress)

print("Steady state stresses:", steady_state_stresses)

# reduced viscosity = (steady state stress) / (eta_f * gamma_dot)
reduced_viscosities = [stress / (eta_f * gamma_dot) for stress in steady_state_stresses]
print("Reduced viscosities:", reduced_viscosities)

In [ ]:
# Creating a clear dataframe with columns: phi, a, reduced_viscosity
data = []
for col, eta_r in zip(new_df.columns[1:], reduced_viscosities):
    m = re.match(r"phi([0-9.]+)_a([0-9.]+)_stress_xy", col)
    if m:
        phi = float(m.group(1))
        a = float(m.group(2))
        data.append({"phi": phi, "a": a, "reduced_viscosity": eta_r})
df_viscosity = pd.DataFrame(data)
print(df_viscosity)

In [ ]:

# tau_corr = 1 strain unit = 1/gamma_dot timesteps = 100 timesteps
# N_eff = N / (2 * tau_steps + 1)
# SEM_corrected = std / sqrt(N_eff)

tau_steps = 1.0 / gamma_dot          # = 100 timesteps

sem_strain = []
for col in new_df.columns[1:]:       # skip Time
    series = new_df[col][200:].dropna()
    N      = len(series)
    n_eff  = max(N / (2 * tau_steps + 1), 1.0)
    std    = series.std(ddof=1)
    sem_strain.append((std / np.sqrt(n_eff)) / (eta_f * gamma_dot))

sem_strain = np.array(sem_strain)
print("SEM (strain-corrected):", sem_strain)
print(f"Correction factor vs naive: {(sem_strain / np.array([new_df[col][200:].sem() / (eta_f * gamma_dot) for col in new_df.columns[1:]])).mean():.1f}x on average")

In [ ]:
phis = sorted({float(re.search(r"phi([0-9.]+)", c).group(1))
               for c in new_df.columns[1:]})
markers = ['o','s','v','^','D','p','*','h','x','+']
colors  = plt.cm.viridis(np.linspace(0, 0.9, len(phis)))  # viridis

phi_styles = {phi: {'marker': markers[i % len(markers)],
                    'color': colors[i]}
              for i, phi in enumerate(phis)}



def log_errorbars(means, sems):
    """Asymmetric log-space error bars so caps are visible on log axes."""
    means = np.asarray(means, dtype=float)
    sems  = np.asarray(sems,  dtype=float)
    rel   = sems / means
    return means * (1 - np.exp(-rel)), means * (np.exp(rel) - 1)

fig, ax = plt.subplots(figsize=(8, 6))
plotted_phis = set()

for idx, col in enumerate(new_df.columns[1:]):
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue
    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx]
    sigma_0 = 2 * np.pi * a
    eta_r   = reduced_viscosities[idx]

    lo, hi = log_errorbars([eta_r], [sem_strain[idx]])

    style = phi_styles[phi]
    label = f"{phi}" if phi not in plotted_phis else None

    ax.errorbar(sigma / sigma_0, eta_r,
                yerr=[[lo[0]], [hi[0]]],
                fmt=style['marker'],
                color=style['color'],
                capsize=3, capthick=1.2, elinewidth=1.0,
                label=label)

    plotted_phis.add(phi)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\sigma / \sigma_a$", fontsize=22, fontweight='bold')
ax.set_ylabel(r"$\eta_r$", fontsize=22, fontweight='bold')
ax.minorticks_on()
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax.tick_params(labelsize=17)
ax.legend(fontsize=13, frameon=False, handlelength=0.5, handletextpad=0.3)
plt.tight_layout()
plt.savefig("flowcurve_strain_uncertainty.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.optimize import curve_fit

for idx, col in enumerate(new_df.columns[1:]):           # skip Time
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue
    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx]
    sigma_0 = 2 * np.pi * a

    style = phi_styles[phi]
    label = f"phi={phi}" if phi not in plotted_phis else None


    plt.scatter((3*0.1*0.01)/a,
                sigma*(3/a),
                color=style['color'],
                marker=style['marker'],
                label=label)

    plotted_phis.add(phi)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("1 / Sigma_0")
plt.ylabel("Stress")
plt.title("Flow Curves: Stress vs Sigma/Sigma_0")
plt.legend()
plt.show()

In [ ]:
from collections import defaultdict


points_by_phi = defaultdict(list)

for idx, col in enumerate(new_df.columns[1:]):  # skip Time
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue

    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx] *(3/a)   # y value (stress)
    sigma_0 = 2 * np.pi * a                # stress scale
    x = (3*0.1*0.01)/a             

    points_by_phi[phi].append((x, sigma))

# fit first 3 smallest-x points for each phi, print intercept 
sigma_y_by_phi = {}

for phi, pts in sorted(points_by_phi.items()):
    pts_sorted = sorted(pts, key=lambda t: t[0])   # sort by x (ascending)
    if len(pts_sorted) < 3:
        print(f"phi={phi}: only {len(pts_sorted)} points -> skipping")
        continue

    xs = np.array([p[0] for p in pts_sorted[:3]])
    ys = np.array([p[1] for p in pts_sorted[:3]])

    # linear fit: y = m x + b
    m_fit, b_fit = np.polyfit(xs, ys, 1)
    sigma_y_by_phi[phi] = b_fit

    print(f"phi={phi}: fit on smallest 3 x -> sigma_y (y-intercept) = {b_fit:.6g}, slope = {m_fit:.6g}")

In [ ]:
from collections import defaultdict
from sklearn.metrics import r2_score

points_by_phi = defaultdict(list)

# First pass: collect data (no fitting yet) 
for idx, col in enumerate(new_df.columns[1:]):  # skip Time
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue
    phi = float(m.group(1))
    a   = float(m.group(2))
    sigma   = steady_state_stresses[idx] *(3/a)
    sigma_0 = 2 * np.pi * a
    x = (3*0.1*0.01)/a
    points_by_phi[phi].append((x, sigma))

# Second pass: plot + fit 
for phi, pts in sorted(points_by_phi.items()):
    style = phi_styles[phi]
    label = f"phi={phi}" if phi not in plotted_phis else None

    pts_sorted = sorted(pts, key=lambda t: t[0])
    xs = np.array([p[0] for p in pts_sorted])
    ys = np.array([p[1] for p in pts_sorted])

    plt.scatter(xs, ys,
                color=style['color'],
                marker=style['marker'],
                label=label)
    plotted_phis.add(phi)

    # Fit first 3 smallest x points
    if len(xs) >= 3:
        xs_fit = xs[:3]
        ys_fit = ys[:3]

        def model(x, m, b):
            return m * x + b

        popt, _ = curve_fit(model, xs_fit, ys_fit)
        m_fit, b_fit = popt

        

        print(f"phi={phi}: sigma_y (y-intercept) = {b_fit:.6g},  R² = {r2:.4f}")

        x_line = np.linspace(0, xs_fit[-1], 200)
        y_line = model(x_line, m_fit, b_fit)
        plt.plot(x_line, y_line,
                 color=style['color'],
                 linestyle='--',
                 label=f"phi={phi}")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("1 / Sigma_0")
plt.ylabel("Stress")
plt.title("Flow Curves with Linear Fits (first 3 smallest x)")
plt.legend()
plt.show()

In [ ]:
# Plot the yield stress (sigma_y) vs phi using the fitted intercepts
plt.figure(figsize=(8, 6))
phis = sorted(sigma_y_by_phi.keys())
sigma_ys = [sigma_y_by_phi[phi] for phi in phis]
plt.plot(phis, sigma_ys, marker='o', linestyle='')
plt.xlabel("Phi")
plt.yscale("log")
plt.ylabel("Yield Stress (sigma_y)")
plt.title("Yield Stress vs Packing Fraction")
plt.show()

df_export = pd.DataFrame({
    "phi" : phis,
    "sigma_y" : sigma_ys
})

df_export.to_csv("1_1_extended_flowcurve.csv", index=False)

In [ ]:
columns_1 = []
column_1_names = []

for folder in folders:
    # phi from folder name: data_1_1_phi0.55
    mphi = re.search(r"phi([0-9.]+)", folder.name)
    if not mphi:
        continue
    phi_str = mphi.group(1)

    run_files = sorted(folder.glob("run_*"))  # e.g. run_0.01.txt etc.
    print(folder.name, "->", len(run_files), "run files")

    for file in run_files:
        start = find_table_start(file)
        end = find_table_end(file)

        df2 = pd.read_csv(
            file,
            delim_whitespace=True,
            skiprows=start,
            nrows=(end - start) if end is not None else None,
            on_bad_lines="skip",
            engine="python"
        )

        col = df2.iloc[:, 7]  # 8th column (contact_number)
        
    

        # change filename: run_0.006.txt -> 0.006
        a_str = file.stem.replace("run_", "")



        new_name = f"phi{phi_str}_a{a_str}_contact_number"
        col = col.rename(new_name)

        columns_1.append(col)
        column_1_names.append(new_name)

new_df2 = pd.concat(columns_1, axis=1)
new_df2.columns = column_1_names
new_df2.insert(0, "Time", range(len(new_df2)))

print("Final dataframe shape:", new_df2.shape)




In [ ]:
steady = new_df2[(new_df2["Time"] >= 100) & (new_df2["Time"] <= 500)]

# mean for every column except Time
steady_means = steady.drop(columns=["Time"]).mean()

print(steady_means)




In [ ]:
records = []
for name, zbar in steady_means.items():
    m = re.search(r"phi([0-9.]+)_a([0-9.eE+-]+)_contact_number", name)
    if not m:
        continue
    phi = float(m.group(1))
    a   = float(m.group(2))
    records.append({"phi": phi, "a": a, "Z_mean": zbar})

tidy = pd.DataFrame(records)

# optional: sort so lines draw nicely
tidy = tidy.sort_values(["a", "phi"])

import pickle
plt.figure(figsize=(10, 4))

for phi in phis:
    style = phi_styles[phi]

    xs = []
    zs = []
    for idx, col in enumerate(new_df.columns[1:]):   # skip Time
        m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
        if not m:
            continue
        if float(m.group(1)) != phi:                # only this φ
            continue
        a = float(m.group(2))

        sigma   = steady_state_stresses[idx]
        sigma_0 = 2 * np.pi * a

        z_mean = tidy[(tidy["phi"] == phi) & (tidy["a"] == a)]["Z_mean"].values[0]
        xs.append(sigma / sigma_0)
        zs.append(z_mean)

    # sort by x so the connecting line is monotonic
    order = np.argsort(xs)
    xs = np.array(xs)[order]
    zs = np.array(zs)[order]

    plt.plot(xs, zs,
             marker=style['marker'],
             color=style['color'],
             linestyle='-',
             label=f"φ={phi}")

plt.xscale("log")
plt.xlabel("$\sigma / \sigma_0$", fontsize=14, fontweight='bold')
plt.ylabel("⟨Z⟩", fontsize=14, fontweight='bold')
plt.yticks(size=14)
plt.xticks(size=14)
plt.legend(fontsize=12)
plt.tight_layout()
fig = plt.gcf()  # grab the current figure
with open('fig3.pkl', 'wb') as f:  # change to fig2.pkl, fig3.pkl
    pickle.dump(fig, f)
plt.show()